# AIMO Solver (v58)-Colab

This notebook is Colab-native and reproducible while preserving the v58 **strategy**:
- parallel attempts
- entropy-based confidence
- inverse-entropy weighted voting
- early stop and single rescue attempt

It intentionally does **not** copy all Kaggle-specific infrastructure (local vLLM server launch, Kaggle gateway hooks, full tool-calling sandbox).

##  Install Dependencies

In [1]:
!pip -q install -U openai transformers accelerate polars sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.0/828.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 MB 12.4 MB/s eta 0:00:00


##  Runtime Probe

In [2]:
import platform
import torch

print('Python:', platform.python_version())
print('Platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.13
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
CUDA available: False


##  Inputs (GPT-OSS-120B Endpoint Only)

This notebook is locked to GPT-OSS-120B via an OpenAI-compatible endpoint for writeup reproducibility.

Default values below are set from Hugging Face Inference Providers docs:
- base URL: https://router.huggingface.co/v1
- model: openai/gpt-oss-120b:fastest

You must provide your own API key/token .

In [6]:
# ---- REQUIRED / USER INPUTS (GPT-OSS-120B endpoint only) ----

# Web-verified OpenAI-compatible defaults from Hugging Face Inference Providers docs.
OPENAI_BASE_URL = 'https://router.huggingface.co/v1'
MODEL_NAME = 'openai/gpt-oss-120b:fastest'

# Paste your Hugging Face token (must have Inference Providers permission).
OPENAI_API_KEY = ''  # example: 'hf_xxxxxxxxxxxxxxxxxxxx'

# v58-style core strategy params
ATTEMPTS = 8
EARLY_STOP = 4
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 1024
TOP_LOGPROBS = 5
WORKERS = 8
SEED = 42

if not OPENAI_API_KEY:
    print('WARNING: OPENAI_API_KEY is empty. Add your HF token before running backend init.')

print('MODEL_NAME =', MODEL_NAME)
print('OPENAI_BASE_URL =', OPENAI_BASE_URL)
print('ATTEMPTS/EARLY_STOP =', ATTEMPTS, '/', EARLY_STOP)

MODEL_NAME = openai/gpt-oss-120b:fastest
OPENAI_BASE_URL = https://router.huggingface.co/v1
ATTEMPTS/EARLY_STOP = 8 / 4


## Core Strategy Implementation (v58-equivalent)

In [7]:
import re
import math
import random
import hashlib
import numpy as np
import pandas as pd
import polars as pl

from dataclasses import dataclass
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Dict, Any, List

from openai import OpenAI
from transformers import set_seed

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class CFG:
    system_prompt: str = (
        'You are an elite mathematical problem solver with expertise at IMO level. '
        'Find the correct answer through rigorous mathematical reasoning.\n\n'
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Rephrase the problem and identify constraints.\n'
        '2. EXPLORE: Consider multiple plausible methods before committing.\n'
        '3. PLAN: Outline key steps.\n'
        '4. EXECUTE: Solve carefully with clear logic.\n'
        '5. VERIFY: Check arithmetic, constraints, and edge behavior.\n\n'
        '# Output Format:\n'
        'Final answer must be an integer in [0, 99999], written as \\boxed{n}.'
    )
    system_prompt_alt: str = (
        'Use a different strategy from your first attempt (algebraic/combinatorial/'
        'number-theoretic/geometric). Verify carefully and return final answer as \\boxed{n} '
        'with n in [0, 99999].'
    )
    preference_prompt: str = (
        'Use careful step-by-step reasoning, compare alternatives when helpful, and '
        'prioritize correctness over brevity. Ensure the final integer answer is explicit.'
    )
    attempts: int = ATTEMPTS
    early_stop: int = EARLY_STOP
    temperature: float = TEMPERATURE
    max_new_tokens: int = MAX_NEW_TOKENS
    top_logprobs: int = TOP_LOGPROBS
    workers: int = WORKERS
    seed: int = SEED

cfg = CFG()

def scan_for_answer(text: str) -> Optional[int]:
    if not text:
        return None
    pattern1 = r'\\boxed\s*\{\s*([0-9,\s]+)\s*\}'
    m1 = re.findall(pattern1, text)
    if m1:
        try:
            v = int(m1[-1].replace(',', '').replace(' ', ''))
            if 0 <= v <= 99999:
                return v
        except ValueError:
            pass

    pattern2 = r'final\s+answer\s+is\s*([0-9,]+)'
    m2 = re.findall(pattern2, text, re.IGNORECASE)
    if m2:
        try:
            v = int(m2[-1].replace(',', ''))
            if 0 <= v <= 99999:
                return v
        except ValueError:
            pass

    pattern3 = r'(?:^|\n|\s)answer\s*[:=]\s*([0-9,]+)(?:\s|$)'
    m3 = re.findall(pattern3, text, re.IGNORECASE)
    if m3:
        try:
            v = int(m3[-1].replace(',', ''))
            if 0 <= v <= 99999:
                return v
        except ValueError:
            pass

    return None

def entropy_from_top_logprobs_dict(top_logprobs_dict: Dict[str, float]) -> float:
    if not top_logprobs_dict:
        return float('inf')
    probs = []
    for _, lp in top_logprobs_dict.items():
        p = math.exp(lp)
        if p > 0:
            probs.append(p)
    if not probs:
        return float('inf')
    z = sum(probs)
    probs = [p / z for p in probs]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def mean_entropy(logprob_entries: List[Dict[str, float]]) -> float:
    vals = [entropy_from_top_logprobs_dict(d) for d in logprob_entries if isinstance(d, dict) and d]
    if not vals:
        return float('inf')
    return float(sum(vals) / len(vals))

In [8]:
class EndpointBackend:
    def __init__(self, base_url: str, api_key: str, model_name: str):
        if not base_url or not api_key:
            raise ValueError('Endpoint mode requires OPENAI_BASE_URL and OPENAI_API_KEY.')
        if 'gpt-oss-120b' not in model_name.lower():
            raise ValueError('This notebook is restricted to GPT-OSS-120B model IDs.')
        self.client = OpenAI(base_url=base_url, api_key=api_key, timeout=180)
        self.model_name = model_name

    def run_attempt(self, prompt: str, system_prompt: str, attempt_seed: int) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt},
            ],
            temperature=cfg.temperature,
            max_tokens=cfg.max_new_tokens,
            logprobs=True,
            top_logprobs=cfg.top_logprobs,
            seed=attempt_seed,
        )
        choice = resp.choices[0]
        text = choice.message.content or ''

        log_entries = []
        if getattr(choice, 'logprobs', None) and getattr(choice.logprobs, 'content', None):
            for token_lp in choice.logprobs.content:
                d = {}
                for tlp in (token_lp.top_logprobs or []):
                    d[getattr(tlp, 'token', '')] = float(getattr(tlp, 'logprob', -100.0))
                if d:
                    log_entries.append(d)

        return {
            'text': text,
            'entropy': mean_entropy(log_entries),
        }

def build_backend():
    return EndpointBackend(OPENAI_BASE_URL, OPENAI_API_KEY, MODEL_NAME)

backend = build_backend()
print('Backend initialized:', type(backend).__name__)
print('Using model:', MODEL_NAME)
print('Using base URL:', OPENAI_BASE_URL)

Backend initialized: EndpointBackend
Using model: openai/gpt-oss-120b:fastest
Using base URL: https://router.huggingface.co/v1


In [9]:
class AIMO3ColabSolver:
    def __init__(self, backend):
        self.backend = backend

    def _select_answer(self, detailed_results: List[Dict[str, Any]]) -> int:
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for row in detailed_results:
            ans = row['Answer']
            ent = row['Entropy']
            if ans is not None:
                w = 1.0 / max(ent, 1e-9)
                answer_weights[ans] += w
                answer_votes[ans] += 1

        scored = [
            {'answer': a, 'votes': answer_votes[a], 'score': s}
            for a, s in answer_weights.items()
        ]
        scored.sort(key=lambda x: x['score'], reverse=True)

        if not scored:
            print('Final Answer: 0')
            return 0

        vote_df = pd.DataFrame(scored).round({'score': 3})
        display(vote_df.rename(columns={'answer': 'Answer', 'votes': 'Votes', 'score': 'Score'}))

        final_answer = int(scored[0]['answer'])
        print(f'Final Answer: {final_answer}')
        return final_answer

    def _process_attempt(self, user_input: str, system_prompt: str, attempt_index: int) -> Dict[str, Any]:
        problem_hash = int(hashlib.blake2s(user_input.encode('utf-8'), digest_size=4).hexdigest(), 16)
        attempt_seed = int((cfg.seed + attempt_index) ** 2) ^ (problem_hash & 0x7FFFFFFF)

        try:
            out = self.backend.run_attempt(user_input, system_prompt, attempt_seed)
            text = out['text']
            answer = scan_for_answer(text)
            entropy = float(out['entropy'])
            return {
                'Attempt': attempt_index + 1,
                'Response Length': len(text),
                'Python Calls': 0,
                'Python Errors': 0,
                'Entropy': entropy,
                'Answer': answer,
            }
        except Exception:
            return {
                'Attempt': attempt_index + 1,
                'Response Length': 0,
                'Python Calls': 0,
                'Python Errors': 1,
                'Entropy': float('inf'),
                'Answer': None,
            }

    def solve_problem(self, problem: str) -> int:
        print(f'Problem: {problem[:180]}...')
        user_input = f"{problem} {cfg.preference_prompt}"

        detailed_results = []
        valid_answers = []

        with ThreadPoolExecutor(max_workers=min(cfg.workers, cfg.attempts)) as ex:
            futures = [ex.submit(self._process_attempt, user_input, cfg.system_prompt, i) for i in range(cfg.attempts)]
            for fut in as_completed(futures):
                result = fut.result()
                detailed_results.append(result)
                if result['Answer'] is not None:
                    valid_answers.append(result['Answer'])

                counts = Counter(valid_answers).most_common(1)
                if counts and counts[0][1] >= cfg.early_stop:
                    break

        if detailed_results:
            res_df = pd.DataFrame(detailed_results)
            res_df['Entropy'] = res_df['Entropy'].round(3)
            res_df['Answer'] = res_df['Answer'].astype('Int64')
            display(res_df)

        if not valid_answers:
            print('No valid answers found. Running one rescue attempt...')
            rescue = self._process_attempt(user_input, cfg.system_prompt_alt, cfg.attempts)
            detailed_results.append(rescue)
            if rescue['Answer'] is not None:
                valid_answers.append(rescue['Answer'])

        if not valid_answers:
            print('No valid answers found.')
            return 0

        return self._select_answer(detailed_results)

solver = AIMO3ColabSolver(backend)
print('Solver ready.')

Solver ready.


##  Smoke Test

In [10]:
sample_problem = 'Find the remainder when 7^2026 is divided by 13.'
ans = solver.solve_problem(sample_problem)
print('Smoke-test answer =', ans)

Problem: Find the remainder when 7^2026 is divided by 13....


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,6,0,0,1,inf,<NA>
1,2,738,0,0,0.230,4
2,3,646,0,0,0.200,4
3,4,621,0,0,0.240,4
4,8,662,0,0,0.199,4


,Answer,Votes,Score
0,4,4,18.565


Final Answer: 4
Smoke-test answer = 4


##  CSV Harness (Reproducible Interface)

Expected input CSV columns: `id`, and either `problem` or `question`.

In [11]:
def run_local_csv(test_path='test.csv', output_path='submission.csv'):
    test_df = pl.read_csv(test_path)
    question_col = 'problem' if 'problem' in test_df.columns else 'question'
    if question_col not in test_df.columns:
        raise ValueError(f'Expected problem/question column in {test_path}')

    rows = []
    for row in test_df.iter_rows(named=True):
        pred = solver.solve_problem(str(row[question_col]))
        rows.append({'id': row['id'], 'answer': int(pred)})

    out_df = pl.DataFrame(rows)
    out_df.write_csv(output_path)
    print(f'Wrote {output_path}')
    return out_df

# Example:
# out = run_local_csv('test.csv', 'submission.csv')
# out.head()

##  Practical Compatibility Notes

- This notebook is endpoint-only and locked to GPT-OSS-120B model IDs.
- It runs on Colab CPU or GPU because inference happens on the remote endpoint.
- For best reproducibility, keep defaults:
  - OPENAI_BASE_URL = https://router.huggingface.co/v1
  - MODEL_NAME = openai/gpt-oss-120b:fastest
- You must provide your own HF token as OPENAI_API_KEY.